# Module 8 — Structured Output and Extraction

Companion notebook to [`docs/modules/08_structured_output_and_extraction.md`](../docs/modules/08_structured_output_and_extraction.md).

Like Modules 6/6.5/7, the pipeline itself is fully built and tested against `FakeRuntime`.
This notebook demonstrates every stage of the reliability ladder live: constrained decoding,
repair retry, deterministic confidence scoring, and human review queueing.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "packages"))
sys.path.insert(0, str(REPO_ROOT / "scripts" / "module_01"))
sys.path.insert(0, str(REPO_ROOT / "scripts" / "module_08"))
print(f"Repo root: {REPO_ROOT}")

Repo root: /Users/bhakti/workspace/local_ai_app


## 1. A clean extraction, end to end

In [2]:
from local_ai_core.extraction.pipeline import ExtractionPipeline
from local_ai_core.extraction.schemas import PersonExtraction
from local_ai_core.runtimes.fake import FakeRuntime

clean_runtime = FakeRuntime(default_response='{"name": "Maria", "age": 29, "city": "Austin"}')
pipeline = ExtractionPipeline(clean_runtime, PersonExtraction, required_fields=["name", "age", "city"])
result = await pipeline.run("Maria moved to Austin last spring. She just turned 29.", "demo-model")

print(f"Parsed: {result.parsed}")
print(f"Confidence: {result.confidence} (deterministic, not model-self-reported)")
print(f"Used constrained decoding: {result.used_constrained_decoding}")
print(f"Needed review: {result.needs_review}")

Parsed: name='Maria' age=29 city='Austin'
Confidence: high (deterministic, not model-self-reported)
Used constrained decoding: True
Needed review: False


## 2. Repair retry: the model fixes its own mistake

In [3]:
class FlakyThenFixedRuntime(FakeRuntime):
    """Returns markdown-wrapped (invalid) JSON on the first call, valid JSON after."""
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self._first_call = True

    async def generate(self, request):
        if self._first_call:
            self._first_call = False
            self.responses = {request.model: '```json\n{"name": "Bob"\n```'}  # malformed on purpose
        else:
            self.responses = {request.model: '{"name": "Bob", "age": 41, "city": "Denver"}'}
        return await super().generate(request)

flaky_runtime = FlakyThenFixedRuntime()
pipeline2 = ExtractionPipeline(flaky_runtime, PersonExtraction, required_fields=["name", "age", "city"])
result2 = await pipeline2.run("Bob moved to Denver. He is 41.", "demo-model")

print(f"Parsed: {result2.parsed}")
print(f"Used repair retry: {result2.used_repair_retry}")
print(f"Confidence: {result2.confidence} (downgraded because a repair was needed)")

Parsed: name='Bob' age=41 city='Denver'
Used repair retry: True
Confidence: medium (downgraded because a repair was needed)


## 3. Missing fields and the review queue

In [4]:
from local_ai_core.extraction.review_queue import ReviewQueue

# Two risk factors at once (missing fields + unconstrained decoding) to
# actually cross the "low confidence" threshold that triggers review -
# one risk factor alone (e.g. missing fields under constrained decoding)
# only downgrades to "medium", which does NOT queue for review by default.
sparse_runtime = FakeRuntime(default_response='{"name": null, "age": null, "city": null}')
queue = ReviewQueue()
pipeline3 = ExtractionPipeline(
    sparse_runtime, PersonExtraction, required_fields=["name", "age", "city"],
    max_repair_attempts=0, review_queue=queue, response_format_type="text",
)
result3 = await pipeline3.run("Contact John immediately.", "demo-model")

print(f"Confidence: {result3.confidence}")
print(f"Needs review: {result3.needs_review}")
print(f"Pending review items: {len(queue)}")
for item in queue.list_pending():
    print(f"  - {item.item_id[:8]}...: {item.reason}")

Confidence: low
Needs review: True
Pending review items: 1
  - 56f17a95...: confidence=low


## 4. Chunked extraction over a long document

In [5]:
long_doc = (
    "Section one talks about the weather and has nothing to do with the person.\n\n"
    "Maria works at the downtown office.\n\n"
    "She recently celebrated turning 29 years old.\n\n"
    "Her city of residence is Austin, Texas.\n\n"
    "The rest of this document is unrelated filler content repeated for length. " * 5
)

chunk_runtime = FakeRuntime(default_response='{"name": "Maria", "age": 29, "city": "Austin"}')
pipeline4 = ExtractionPipeline(chunk_runtime, PersonExtraction, required_fields=["name", "age", "city"])
chunked_result = await pipeline4.run_chunked(long_doc, "demo-model", max_chars=150)

print(f"Document length: {len(long_doc)} chars")
print(f"Runtime calls made (= number of chunks processed): {chunk_runtime.call_count}")
print(f"Merged fields: {chunked_result.fields}")

Document length: 1380 chars
Runtime calls made (= number of chunks processed): 11
Merged fields: {'name': 'Maria', 'age': 29, 'city': 'Austin'}


## 5. Lab 8: prompt-only vs. JSON-schema vs. grammar, against fakes with realistic capability differences

Mirrors the real capability differences Module 5's `feature_matrix.py` documented: this fake
supports JSON schema but not grammar (like Ollama), and is less reliable in prompt-only mode
(matching Module 1 §11's small-model behavior).

In [6]:
import constrained_decoding_runner as lab8
from local_ai_core.runtimes.errors import FeatureNotSupported
from IPython.display import Markdown, display

class RealisticRuntime(FakeRuntime):
    """json_schema: reliable. grammar: unsupported (like Ollama). text: occasionally malformed."""
    async def generate(self, request):
        if request.response_format.type == "grammar":
            raise FeatureNotSupported("this fake has no grammar support, like real Ollama")
        if request.response_format.type == "text" and "Priya" in request.prompt:
            self.responses = {request.model: "Here you go: Priya is in Chicago I think."}  # not JSON
        else:
            self.responses = {request.model: '{"name": "Priya", "age": null, "city": "Chicago"}' if "Priya" in request.prompt else '{"name": "Maria", "age": 29, "city": "Austin"}'}
        return await super().generate(request)

cases = lab8.load_matching_golden_cases()
results = await lab8.run_lab(RealisticRuntime(), "demo-model", cases)

display(Markdown(lab8.results_to_markdown_table(results)))

| Mode | Invalid JSON rate | Field accuracy | p95 latency (s) | Constrained decoding used |
|---|---:|---:|---:|---:|
| text | 50% | 100% | 0.000 | 0% |
| json_schema | 0% | 100% | 0.000 | 100% |
| grammar | 50% | 100% | 0.000 | 0% |

## 6. Real 3-way comparison and golden-label evaluation, if available

Expected to skip on this machine (no local model runtime installed by design).

In [7]:
from ollama_probe import is_ollama_available

if is_ollama_available():
    from local_ai_core.runtimes.ollama import OllamaRuntime
    real_runtime = OllamaRuntime()
    real_results = await lab8.run_lab(real_runtime, "qwen2.5:1.5b", cases)
    display(Markdown(lab8.results_to_markdown_table(real_results)))
    await real_runtime.aclose()
else:
    print("SKIPPED: Ollama is not reachable at http://localhost:11434.")
    print("This machine deliberately has no local model runtime installed (see PROGRESS.md).")
    print("On the resourced 32GB Mac: re-run this cell for a real 3-way comparison.")

SKIPPED: Ollama is not reachable at http://localhost:11434.
This machine deliberately has no local model runtime installed (see PROGRESS.md).
On the resourced 32GB Mac: re-run this cell for a real 3-way comparison.


## 7. Take it to the command line

```bash
uv run python scripts/module_08/constrained_decoding_runner.py --model qwen2.5:1.5b
uv run python scripts/module_08/extraction_eval.py --model qwen2.5:1.5b
```

Then fold the output into
[`reports/module_08_structured_output_reliability_report.md`](../reports/module_08_structured_output_reliability_report.md).